In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import rasterio
from rasterio import plot
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
import math
from skimage.measure import block_reduce
from sklearn.metrics import f1_score, precision_score, recall_score
from scipy.interpolate import griddata

In [6]:
from utils.site import Site
from utils.tree import Tree
from utils.constants import SILVER_FOLDERPATH, ETHZ_COCOA_MAP_UINT8_FILEPATH, ETHZ_COCOA_MASK_FILEPATH

In [7]:
with rasterio.open(ETHZ_COCOA_MAP_UINT8_FILEPATH) as src:
    print("Boundaries of the map")
    print("Lon:", src.bounds.left, "-->", src.bounds.right)
    print("Lat:", src.bounds.bottom, "-->", src.bounds.top)

    # Allocate empty uint8 array 
    data = np.empty(src.shape, dtype=np.uint8)
    
    # Iterate through the file block-by-block to avoid RAM overload
    n_window = len(list(src.block_windows(bidx=1)))
    for idx, (_, window) in enumerate(src.block_windows(bidx=1)):
        print(f"Processing: {idx+1}/{n_window}", end="\r")
        chunk = src.read(indexes=1, window=window, masked=True)
        data[window.toslices()] = chunk

print(f"Memory used by array: {data.nbytes / (1024**3):.2f} GB")

Boundaries of the map
Lon: -8.59923 --> 1.1916899999999995
Lat: 4.36182 --> 11.17328
Memory used by array: 7.58 GB


In [8]:
COCOA_THRESH = 90
NO_COCOA_THRESH = 10

nodata_mask = data == 255
cocoa_mask = (data >= COCOA_THRESH) & ~nodata_mask
nococoa_mask = (data <= NO_COCOA_THRESH) & ~nodata_mask

In [ ]:
# Compile masks
data[:] = 0
data[nococoa_mask] = 1
data[cocoa_mask] = 2

In [ ]:
# Plot map
scale_factor = 50
plt.imshow(data[::scale_factor, ::scale_factor])
plt.title(f"ETHZ Cocoa map, scaled down")

In [ ]:
# Dump to disk
with rasterio.open(ETHZ_COCOA_MAP_UINT8_FILEPATH) as src:
    profile = src.profile

profile.update(dtype=data.dtype, count=1, compress='lzw')

with rasterio.open(ETHZ_COCOA_MASK_FILEPATH, 'w', **profile) as dst:
    dst.write(data, 1)

print(f"Dumped: {ETHZ_COCOA_MASK_FILEPATH}")